<a href="https://colab.research.google.com/github/nickkhymenko/asteria-credit-risk-ml/blob/main/02_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — 1. Data Understanding

**Project:** Asteria Finance — Repayment Difficulty Prediction

**Objective:** Explore the structure of the relational credit dataset, identify relationships between tables, assess data quality, and determine potential features for machine learning.

This notebook focuses on data understanding. No predictive models are trained at this stage.

In [10]:

import pandas as pd

applications = pd.read_csv("/content/loan_applications.csv")
previous = pd.read_csv("/content/previous_loan_applications.csv")

print("Main applications:", applications.shape)
print("Previous applications:", previous.shape)


Main applications: (27000, 122)
Previous applications: (124037, 37)


## 2. Previous Loan Applications

### 2.1 Dataset Structure and Applicant Identifiers

This section examines the relationship between current applicants and their historical loan applications, including applicant coverage and the number of previous applications per customer.

In [11]:

previous.head()


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN
1,1676258,433469,Cash loans,22242.825,247500.0,268083.0,0.0,247500.0,THURSDAY,14,...,Consumer electronics,18.0,high,Cash X-Sell: high,365243.0,-1853.0,-1343.0,-1343.0,-1334.0,1.0
2,2138466,170289,Consumer loans,3286.485,32562.0,32562.0,0.0,32562.0,FRIDAY,12,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-273.0,57.0,-183.0,-179.0,0.0
3,1629736,255331,Revolving loans,10125.000,202500.0,202500.0,NaN,202500.0,SATURDAY,14,...,XNA,0.0,XNA,Card X-Sell,-265.0,-234.0,365243.0,365243.0,365243.0,0.0
4,2740958,281589,Consumer loans,6752.250,35955.0,33115.5,4500.0,35955.0,THURSDAY,9,...,Connectivity,6.0,high,POS mobile with interest,365243.0,-952.0,-802.0,-802.0,-783.0,0.0


In [3]:

print("Unique clients in main table:",
      applications["SK_ID_CURR"].nunique())

print("Unique clients in previous applications:",
      previous["SK_ID_CURR"].nunique())

print("Unique previous applications:",
      previous["SK_ID_PREV"].nunique())

print("\nPrevious applications per client:")
print(previous["SK_ID_CURR"].value_counts().describe())


Unique clients in main table: 27000
Unique clients in previous applications: 25526
Unique previous applications: 124037

Previous applications per client:
count    25526.000000
mean         4.859242
std          4.156504
min          1.000000
25%          2.000000
50%          4.000000
75%          6.000000
max         55.000000
Name: count, dtype: float64


In [4]:

main_ids = set(applications["SK_ID_CURR"])
previous_ids = set(previous["SK_ID_CURR"])

common_ids = main_ids.intersection(previous_ids)

print("Clients in both tables:", len(common_ids))

print("Main clients without previous history:",
      len(main_ids - previous_ids))

print("Previous-history clients not in main table:",
      len(previous_ids - main_ids))

print("Coverage of main applicants:",
      round(len(common_ids) / len(main_ids) * 100, 2), "%")


Clients in both tables: 25526
Main clients without previous history: 1474
Previous-history clients not in main table: 0
Coverage of main applicants: 94.54 %


In [5]:

print("Previous loan types:")
print(previous["NAME_CONTRACT_TYPE"].value_counts(dropna=False))

print("\nPrevious application statuses:")
print(previous["NAME_CONTRACT_STATUS"].value_counts(dropna=False))


Previous loan types:
NAME_CONTRACT_TYPE
Consumer loans     55290
Cash loans         54634
Revolving loans    14084
XNA                   29
Name: count, dtype: int64

Previous application statuses:
NAME_CONTRACT_STATUS
Approved        77990
Canceled        22646
Refused         21314
Unused offer     2087
Name: count, dtype: int64


In [6]:

# Count refused previous applications per client
refused_counts = (
    previous[previous["NAME_CONTRACT_STATUS"] == "Refused"]
    .groupby("SK_ID_CURR")
    .size()
    .rename("PREV_REFUSED_COUNT")
)

# Add the feature to a temporary analysis table
analysis = applications[["SK_ID_CURR", "TARGET"]].copy()

analysis = analysis.merge(
    refused_counts,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

analysis["PREV_REFUSED_COUNT"] = (
    analysis["PREV_REFUSED_COUNT"].fillna(0)
)

# Compare repayment difficulty rates
analysis["HAD_PREVIOUS_REFUSAL"] = (
    analysis["PREV_REFUSED_COUNT"] > 0
)

print(
    analysis.groupby("HAD_PREVIOUS_REFUSAL")["TARGET"]
    .agg(["count", "mean"])
)


                      count      mean
HAD_PREVIOUS_REFUSAL                 
False                 18219  0.069982
True                   8781  0.103063


In [7]:

# Group applicants by number of previous refusals
analysis["REFUSAL_GROUP"] = pd.cut(
    analysis["PREV_REFUSED_COUNT"],
    bins=[-1, 0, 1, 2, 5, float("inf")],
    labels=["0", "1", "2", "3–5", "6+"]
)

refusal_risk = (
    analysis.groupby("REFUSAL_GROUP", observed=True)["TARGET"]
    .agg(["count", "mean"])
)

refusal_risk["mean"] = refusal_risk["mean"] * 100

print(refusal_risk)


               count       mean
REFUSAL_GROUP                  
0              18219   6.998189
1               4128   8.091085
2               1930  10.829016
3–5             2031  12.900049
6+               692  14.450867


In [8]:

# Count all previous applications per client
total_counts = (
    previous.groupby("SK_ID_CURR")
    .size()
    .rename("PREV_TOTAL_COUNT")
)

# Add total application counts
analysis = analysis.merge(
    total_counts,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

# Calculate refusal rate
analysis["PREV_REFUSAL_RATE"] = (
    analysis["PREV_REFUSED_COUNT"]
    / analysis["PREV_TOTAL_COUNT"]
)

# Group applicants by refusal rate
analysis["REFUSAL_RATE_GROUP"] = pd.cut(
    analysis["PREV_REFUSAL_RATE"],
    bins=[-0.01, 0, 0.25, 0.5, 1],
    labels=["0%", "0–25%", "25–50%", "50–100%"]
)

refusal_rate_risk = (
    analysis.groupby("REFUSAL_RATE_GROUP", observed=True)["TARGET"]
    .agg(["count", "mean"])
)

refusal_rate_risk["mean"] *= 100

print(refusal_rate_risk)


                    count       mean
REFUSAL_RATE_GROUP                  
0%                  16745   7.088683
0–25%                4208   8.174905
25–50%               3615  11.037344
50–100%               958  16.910230


## Key Findings — Previous Loan Applications

The previous loan applications dataset contains 124,037 records associated with 25,526 unique applicants. Historical application data is available for 94.54% of applicants in the main dataset, with an average of 4.86 previous applications per applicant with recorded history.

The data exhibits a one-to-many relationship with the main applications table, meaning that historical records must be aggregated at the applicant level before integration.

Previous application decisions show a potential association with repayment difficulty. Applicants without recorded previous refusals have a repayment difficulty rate of approximately 7.00%, compared with 10.31% among applicants with at least one refusal.

A stronger pattern emerges when considering refusal frequency. Repayment difficulty rates increase from 7.09% among applicants with a 0% refusal rate to 16.91% among those with refusal rates above 50%.

**Potential features for modelling:**
- `PREV_TOTAL_COUNT` — total number of previous applications.
- `PREV_REFUSED_COUNT` — number of previously refused applications.
- `PREV_REFUSAL_RATE` — proportion of previous applications refused.
- `HAD_PREVIOUS_REFUSAL` — indicator of at least one recorded refusal.

These relationships are descriptive and do not establish causality or predictive performance. Feature availability at the time of the current application must also be verified to avoid data leakage.

Applicants without recorded historical applications should be distinguished from applicants with recorded applications but no refusals.

## 3. External Credit Accounts

### 3.1 Dataset Structure

This section explores external credit accounts reported for applicants, including dataset structure, applicant coverage, and potentially useful indicators of credit exposure and repayment behaviour.

In [12]:

external = pd.read_csv("/content/external_credit_accounts.csv")

print("Dataset shape:", external.shape)

print("\nUnique applicants:",
      external["SK_ID_CURR"].nunique())

print("\nColumn names:")
print(external.columns.tolist())

display(external.head())


Dataset shape: (128059, 17)

Unique applicants: 23088

Column names:
['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,222183,5714491,Active,currency 1,-784,0,1008.0,NaN,0.00,0,0.0,-411.615,411.615,0.0,Credit card,-694,NaN
1,222183,5714492,Active,currency 1,-774,0,625.0,NaN,NaN,0,127840.5,0.000,0.000,0.0,Credit card,-210,NaN
2,222183,5714493,Active,currency 1,-395,0,1431.0,NaN,NaN,0,1350000.0,1185493.500,0.000,0.0,Consumer credit,-24,NaN
3,222183,5714495,Closed,currency 1,-2744,0,-2561.0,-2559.0,310.50,0,18157.5,NaN,NaN,0.0,Consumer credit,-2559,NaN
4,222183,5714496,Closed,currency 1,-1103,0,-7.0,-343.0,20493.27,0,675000.0,0.000,0.000,0.0,Consumer credit,-343,NaN


In [13]:

# Check external credit account coverage
main_ids = set(applications["SK_ID_CURR"])
external_ids = set(external["SK_ID_CURR"])

common_ids = main_ids.intersection(external_ids)

print("Clients in both tables:", len(common_ids))

print("Main clients without external credit records:",
      len(main_ids - external_ids))

print("External clients not in main table:",
      len(external_ids - main_ids))

print("Coverage:",
      round(len(common_ids) / len(main_ids) * 100, 2), "%")

# Number of external credit accounts per client
print("\nExternal accounts per client:")
print(external["SK_ID_CURR"].value_counts().describe())

# Check unique external account IDs
print("\nUnique bureau account IDs:",
      external["SK_ID_BUREAU"].nunique())


Clients in both tables: 23088
Main clients without external credit records: 3912
External clients not in main table: 0
Coverage: 85.51 %

External accounts per client:
count    23088.000000
mean         5.546561
std          4.368869
min          1.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         50.000000
Name: count, dtype: float64

Unique bureau account IDs: 128059


In [14]:

# Distribution of external credit account statuses
print("External credit account statuses:")
print(external["CREDIT_ACTIVE"].value_counts(dropna=False))

# Overview of overdue days
print("\nOverdue days statistics:")
print(external["CREDIT_DAY_OVERDUE"].describe())

# Accounts with recorded overdue days
overdue_accounts = (external["CREDIT_DAY_OVERDUE"] > 0).sum()

print("\nAccounts with overdue days:", overdue_accounts)

print("Percentage of accounts with overdue days:",
      round(overdue_accounts / len(external) * 100, 2), "%")


External credit account statuses:
CREDIT_ACTIVE
Closed      80038
Active      47534
Sold          486
Bad debt        1
Name: count, dtype: int64

Overdue days statistics:
count    128059.000000
mean          0.854528
std          37.986358
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        2688.000000
Name: CREDIT_DAY_OVERDUE, dtype: float64

Accounts with overdue days: 337
Percentage of accounts with overdue days: 0.26 %


In [15]:

# Identify accounts with recorded overdue days
external["HAS_OVERDUE"] = (
    external["CREDIT_DAY_OVERDUE"] > 0
)

# Aggregate overdue information per applicant
external_overdue = (
    external.groupby("SK_ID_CURR")
    .agg(
        EXT_ACCOUNT_COUNT=("SK_ID_BUREAU", "count"),
        EXT_OVERDUE_COUNT=("HAS_OVERDUE", "sum"),
        EXT_MAX_OVERDUE_DAYS=("CREDIT_DAY_OVERDUE", "max")
    )
    .reset_index()
)

# Create a temporary applicant-level analysis table
external_analysis = applications[
    ["SK_ID_CURR", "TARGET"]
].copy()

external_analysis = external_analysis.merge(
    external_overdue,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

# Keep applicants without external records separate
external_analysis["HAS_EXTERNAL_HISTORY"] = (
    external_analysis["EXT_ACCOUNT_COUNT"].notna()
)

external_analysis["HAS_EXTERNAL_OVERDUE"] = (
    external_analysis["EXT_OVERDUE_COUNT"] > 0
)

# Compare repayment difficulty rates
result = (
    external_analysis.groupby(
        ["HAS_EXTERNAL_HISTORY", "HAS_EXTERNAL_OVERDUE"]
    )["TARGET"]
    .agg(["count", "mean"])
)

result["mean"] *= 100

print(result)


                                           count       mean
HAS_EXTERNAL_HISTORY HAS_EXTERNAL_OVERDUE                  
False                False                  3912  10.250511
True                 False                 22783   7.584603
                     True                    305  16.721311


### Key Findings — External Credit Accounts

The external credit accounts dataset contains 128,059 records covering 23,088 unique applicants, representing 85.51% of the main dataset.

Each external credit account has a unique `SK_ID_BUREAU`, while applicants may have multiple accounts. This confirms a one-to-many relationship requiring applicant-level aggregation before modelling.

Approximately 62.5% of accounts are closed and 37.1% are active. Only 337 accounts (0.26%) have positive recorded overdue days.

**Repayment difficulty differs across applicant groups:**

- Applicants without external credit history: 10.25%.
- Applicants with external history but no recorded current overdue days: 7.58%.
- Applicants with at least one account showing overdue days: 16.72%.

These findings suggest that external credit history and recorded overdue information may provide useful predictive features.

**Potential features for modelling:**
- `EXT_ACCOUNT_COUNT` — number of external credit accounts.
- `EXT_OVERDUE_COUNT` — number of accounts with recorded overdue days.
- `EXT_MAX_OVERDUE_DAYS` — maximum recorded overdue days.
- `HAS_EXTERNAL_HISTORY` — indicator of available external credit history.

The overdue group contains only 305 applicants, so results should be interpreted cautiously. These associations do not establish causality or predictive performance. Feature availability at the application date must be verified to avoid data leakage.

## 4. External Account History

### 4.1 Dataset Structure

This section examines the monthly history of external credit accounts, focusing on account-level records, repayment status information, and relationships with the external credit accounts dataset.

In [16]:

history = pd.read_csv("/content/external_account_history.csv")

print("Dataset shape:", history.shape)

print("\nUnique bureau accounts:",
      history["SK_ID_BUREAU"].nunique())

print("\nColumn names:")
print(history.columns.tolist())

display(history.head())


Dataset shape: (1301123, 3)

Unique bureau accounts: 46196

Column names:
['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5213107,0,C
1,5213107,-1,C
2,5213107,-2,C
3,5213107,-3,C
4,5213107,-4,C


In [17]:

# Distribution of monthly credit statuses
print("Monthly credit status distribution:")
print(history["STATUS"].value_counts(dropna=False))

# Monthly history coverage
print("\nMonths balance statistics:")
print(history["MONTHS_BALANCE"].describe())

# Check whether all history accounts exist in external accounts
history_ids = set(history["SK_ID_BUREAU"])
external_account_ids = set(external["SK_ID_BUREAU"])

print("\nAccounts present in both tables:",
      len(history_ids & external_account_ids))

print("History accounts not found in external table:",
      len(history_ids - external_account_ids))

print("External accounts without monthly history:",
      len(external_account_ids - history_ids))


Monthly credit status distribution:
STATUS
C    627718
0    407085
X    245489
1     14478
5      4092
2      1415
3       524
4       322
Name: count, dtype: int64

Months balance statistics:
count    1.301123e+06
mean    -2.876166e+01
std      2.212278e+01
min     -9.600000e+01
25%     -4.300000e+01
50%     -2.400000e+01
75%     -1.000000e+01
max      0.000000e+00
Name: MONTHS_BALANCE, dtype: float64

Accounts present in both tables: 46196
History accounts not found in external table: 0
External accounts without monthly history: 81863


In [18]:

# Define historical delinquency statuses
delinquency_statuses = ["1", "2", "3", "4", "5"]

# Identify months with recorded delinquency
history["HAS_DELINQUENCY"] = (
    history["STATUS"].astype(str).isin(delinquency_statuses)
)

# Aggregate monthly history to account level
account_history = (
    history.groupby("SK_ID_BUREAU")
    .agg(
        MONTHS_RECORDED=("STATUS", "count"),
        DELINQUENT_MONTHS=("HAS_DELINQUENCY", "sum"),
        EVER_DELINQUENT=("HAS_DELINQUENCY", "max")
    )
    .reset_index()
)

print("Accounts with monthly history:",
      len(account_history))

print("Accounts with historical delinquency:",
      account_history["EVER_DELINQUENT"].sum())

print("Percentage with historical delinquency:",
      round(account_history["EVER_DELINQUENT"].mean() * 100, 2), "%")

print("\nRecorded months per account:")
print(account_history["MONTHS_RECORDED"].describe())


Accounts with monthly history: 46196
Accounts with historical delinquency: 6152
Percentage with historical delinquency: 13.32 %

Recorded months per account:
count    46196.000000
mean        28.165274
std         22.043287
min          1.000000
25%         11.000000
50%         22.000000
75%         40.000000
max         96.000000
Name: MONTHS_RECORDED, dtype: float64


In [19]:

# Link account-level history to applicant IDs
history_clients = account_history.merge(
    external[["SK_ID_BUREAU", "SK_ID_CURR"]],
    on="SK_ID_BUREAU",
    how="left",
    validate="one_to_one"
)

# Aggregate historical delinquency to applicant level
client_history = (
    history_clients.groupby("SK_ID_CURR")
    .agg(
        HIST_ACCOUNT_COUNT=("SK_ID_BUREAU", "count"),
        HIST_DELINQUENT_ACCOUNT_COUNT=("EVER_DELINQUENT", "sum"),
        HIST_TOTAL_DELINQUENT_MONTHS=("DELINQUENT_MONTHS", "sum")
    )
    .reset_index()
)

# Merge with the main applicant dataset
history_analysis = applications[
    ["SK_ID_CURR", "TARGET"]
].merge(
    client_history,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

# Distinguish missing history from no delinquency
history_analysis["HISTORY_GROUP"] = "No monthly history"

has_history = history_analysis["HIST_ACCOUNT_COUNT"].notna()

history_analysis.loc[
    has_history, "HISTORY_GROUP"
] = "History, no delinquency"

history_analysis.loc[
    history_analysis["HIST_DELINQUENT_ACCOUNT_COUNT"] > 0,
    "HISTORY_GROUP"
] = "History with delinquency"

# Compare repayment difficulty rates
history_risk = (
    history_analysis.groupby("HISTORY_GROUP")["TARGET"]
    .agg(["count", "mean"])
)

history_risk["mean"] *= 100

print(history_risk)


                          count       mean
HISTORY_GROUP                             
History with delinquency   2781  10.715570
History, no delinquency    5433   7.454445
No monthly history        18786   7.862238


### Key Finding — External Credit Account History

Monthly credit history contains 1,301,123 observations covering 46,196 external credit accounts. Of these accounts, 6,152 (13.32%) have at least one recorded month with a delinquency status (1–5).

After aggregating historical records to the applicant level, repayment difficulty rates differ across groups:

- **10.72%** for applicants with recorded historical delinquency (n = 2,781).
- **7.45%** for applicants with available monthly history but no recorded delinquency (n = 5,433).
- **7.86%** for applicants without available monthly history (n = 18,786).

Applicants with recorded historical delinquency have approximately **1.44 times the observed repayment difficulty rate** of applicants with available history and no delinquency.

**Implication for modelling:** Historical delinquency indicators, delinquent-month counts and the availability of credit history are promising candidate features. Their predictive contribution should be assessed through validation rather than assumed from descriptive associations.

## 5. Installment Payments — Payment Behaviour

### 5.1 Dataset Structure

This section examines historical installment payment records to understand repayment behaviour, identify potential payment delays, and assess the dataset's suitability for credit risk prediction.

The objective is to explore the relationship between scheduled and actual payments and identify potential features for subsequent modelling.

In [23]:

# Load installment payments dataset
installments = pd.read_csv("/content/instalment_payments.csv")

print("Dataset shape:", installments.shape)

print("\nUnique applicants:",
      installments["SK_ID_CURR"].nunique())

print("\nColumn names:")
print(installments.columns.tolist())

print("\nFirst five rows:")
display(installments.head())


Dataset shape: (1021501, 8)

Unique applicants: 25575

Column names:
['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']

First five rows:


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
1,1570206,147645,0.0,20,-761.0,-761.0,21.510,21.510
2,1434401,173001,5.0,5,-211.0,-217.0,4802.130,4802.130
3,2379431,102345,0.0,84,-1389.0,-1396.0,8100.000,8100.000
4,1362611,196368,1.0,3,-932.0,-949.0,6412.005,6412.005


In [24]:

# Check missing values in payment-related columns
payment_columns = [
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT",
    "AMT_PAYMENT"
]

print("Missing values (%):")
print((installments[payment_columns].isna().mean() * 100).round(2))

# Calculate payment delay in days
installments["PAYMENT_DELAY"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
)

# Identify late payments
installments["IS_LATE"] = installments["PAYMENT_DELAY"] > 0

print("\nPayment delay statistics:")
print(installments["PAYMENT_DELAY"].describe())

print("\nLate payments:")
print("Number:", installments["IS_LATE"].sum())

print("Percentage of all records:",
      round(installments["IS_LATE"].mean() * 100, 2), "%")


Missing values (%):
DAYS_INSTALMENT       0.00
DAYS_ENTRY_PAYMENT    0.02
AMT_INSTALMENT        0.00
AMT_PAYMENT           0.02
dtype: float64

Payment delay statistics:
count    1.021264e+06
mean    -8.765839e+00
std      2.570736e+01
min     -6.680000e+02
25%     -1.400000e+01
50%     -6.000000e+00
75%      0.000000e+00
max      2.780000e+03
Name: PAYMENT_DELAY, dtype: float64

Late payments:
Number: 88392
Percentage of all records: 8.65 %


In [25]:

# Use only records with a known payment date
valid_payments = installments.dropna(
    subset=["DAYS_ENTRY_PAYMENT", "DAYS_INSTALMENT"]
).copy()

# Aggregate payment behaviour to applicant level
payment_features = (
    valid_payments.groupby("SK_ID_CURR")
    .agg(
        TOTAL_PAYMENTS=("IS_LATE", "count"),
        LATE_PAYMENTS=("IS_LATE", "sum"),
        LATE_PAYMENT_RATE=("IS_LATE", "mean"),
        MAX_PAYMENT_DELAY=("PAYMENT_DELAY", "max")
    )
    .reset_index()
)

payment_features["LATE_PAYMENT_RATE"] *= 100

print("Applicants with payment history:",
      len(payment_features))

print("\nLate payment rate statistics (%):")
print(payment_features["LATE_PAYMENT_RATE"].describe())

print("\nFirst five applicants:")
display(payment_features.head())


Applicants with payment history: 25575

Late payment rate statistics (%):
count    25575.000000
mean         7.589394
std         11.510594
min          0.000000
25%          0.000000
50%          1.960784
75%         11.111111
max        100.000000
Name: LATE_PAYMENT_RATE, dtype: float64

First five applicants:


,SK_ID_CURR,TOTAL_PAYMENTS,LATE_PAYMENTS,LATE_PAYMENT_RATE,MAX_PAYMENT_DELAY
0,100008,35,1,2.857143,1317.0
1,100048,67,1,1.492537,6.0
2,100054,47,3,6.382979,9.0
3,100080,63,6,9.523810,4.0
4,100100,177,18,10.169492,20.0


In [26]:

# Merge payment features with the main dataset
payment_analysis = applications[
    ["SK_ID_CURR", "TARGET"]
].merge(
    payment_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

# Create groups based on late payment rate
payment_analysis["LATE_RATE_GROUP"] = pd.cut(
    payment_analysis["LATE_PAYMENT_RATE"],
    bins=[-0.01, 0, 5, 15, 30, 100],
    labels=["0%", "0–5%", "5–15%", "15–30%", "30–100%"]
)

# Compare repayment difficulty across groups
payment_risk = (
    payment_analysis.groupby(
        "LATE_RATE_GROUP", observed=True
    )["TARGET"]
    .agg(["count", "mean"])
)

payment_risk["mean"] *= 100

print("Repayment difficulty by late payment rate:")
print(payment_risk)

print("\nApplicants without payment history:",
      payment_analysis["LATE_PAYMENT_RATE"].isna().sum())


Repayment difficulty by late payment rate:
                 count       mean
LATE_RATE_GROUP                  
0%               11893   6.785504
0–5%              3533   7.245967
5–15%             5296   9.157855
15–30%            3289  10.246275
30–100%           1564  13.491049

Applicants without payment history: 1425


### Key Finding — Installment Payment Behaviour

Historical payment delays show a clear positive association with repayment difficulty.

Applicants with no recorded late payments have a repayment difficulty rate of **6.79%**, compared with **13.49%** among applicants whose late payment rate exceeds 30%.

This suggests that historical repayment behaviour may provide valuable predictive information for credit risk modelling.

Potential features include:
- `TOTAL_PAYMENTS`
- `LATE_PAYMENTS`
- `LATE_PAYMENT_RATE`
- `MAX_PAYMENT_DELAY`

Additionally, 1,425 applicants have no available installment payment history. This missing history should be explicitly accounted for during feature engineering.

These relationships are descriptive and will require validation through predictive modelling.

## 6. POS/Cash Balance Analysis

In [27]:

# Load POS/Cash balance dataset
pos = pd.read_csv("/content/pos_cash_balance.csv")

print("Dataset shape:", pos.shape)

print("\nUnique applicants:",
      pos["SK_ID_CURR"].nunique())

print("\nColumn names:")
print(pos.columns.tolist())

print("\nFirst five rows:")
display(pos.head())


Dataset shape: (754688, 8)

Unique applicants: 25394

Column names:
['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']

First five rows:


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1715348,367990,-33,36.0,35.0,Active,0,0
1,2371489,274851,-32,24.0,16.0,Active,0,0
2,1258390,278261,-32,36.0,36.0,Active,0,0
3,1894403,266542,-37,48.0,40.0,Active,0,0
4,1367593,336653,-31,18.0,18.0,Signed,0,0


In [28]:

# Examine missing values
print("Missing values (%):")
print((pos.isnull().mean() * 100).round(2))

# Examine monthly overdue days
print("\nDays past due statistics:")
print(pos["SK_DPD"].describe())

# Identify records with overdue payments
pos["HAS_OVERDUE"] = pos["SK_DPD"] > 0

print("\nRecords with overdue payments:")
print("Number:", pos["HAS_OVERDUE"].sum())

print("Percentage:",
      round(pos["HAS_OVERDUE"].mean() * 100, 2), "%")

# Examine contract statuses
print("\nContract statuses:")
print(pos["NAME_CONTRACT_STATUS"].value_counts())


Missing values (%):
SK_ID_PREV               0.00
SK_ID_CURR               0.00
MONTHS_BALANCE           0.00
CNT_INSTALMENT           0.25
CNT_INSTALMENT_FUTURE    0.25
NAME_CONTRACT_STATUS     0.00
SK_DPD                   0.00
SK_DPD_DEF               0.00
dtype: float64

Days past due statistics:
count    754688.000000
mean         10.095494
std         118.946141
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        3804.000000
Name: SK_DPD, dtype: float64

Records with overdue payments:
Number: 22047
Percentage: 2.92 %

Contract statuses:
NAME_CONTRACT_STATUS
Active                   690964
Completed                 55814
Signed                     6496
Demand                      656
Returned to the store       420
Approved                    338
Name: count, dtype: int64


In [29]:

# Aggregate POS/Cash history at applicant level
pos_features = (
    pos.groupby("SK_ID_CURR")
    .agg(
        POS_TOTAL_RECORDS=("SK_DPD", "size"),
        POS_OVERDUE_RECORDS=("HAS_OVERDUE", "sum"),
        POS_MAX_DPD=("SK_DPD", "max")
    )
    .reset_index()
)

# Calculate percentage of overdue monthly records
pos_features["POS_OVERDUE_RATE"] = (
    pos_features["POS_OVERDUE_RECORDS"]
    / pos_features["POS_TOTAL_RECORDS"] * 100
)

print("Applicants with POS/Cash history:", len(pos_features))
print("\nFirst five applicants:")
display(pos_features.head())


Applicants with POS/Cash history: 25394

First five applicants:


,SK_ID_CURR,POS_TOTAL_RECORDS,POS_OVERDUE_RECORDS,POS_MAX_DPD,POS_OVERDUE_RATE
0,100008,83,43,1294,51.807229
1,100048,16,0,0,0.000000
2,100054,49,0,0,0.000000
3,100080,66,0,0,0.000000
4,100100,62,0,0,0.000000


In [30]:

# Create a temporary analysis table
pos_analysis = applications[["SK_ID_CURR", "TARGET"]].copy()

# Merge aggregated POS/Cash features
pos_analysis = pos_analysis.merge(
    pos_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

# Separate applicants into three groups
pos_analysis["POS_RISK_GROUP"] = "No POS/Cash history"

has_history = pos_analysis["POS_TOTAL_RECORDS"].notna()

pos_analysis.loc[
    has_history & (pos_analysis["POS_OVERDUE_RECORDS"] == 0),
    "POS_RISK_GROUP"
] = "History, no overdue"

pos_analysis.loc[
    has_history & (pos_analysis["POS_OVERDUE_RECORDS"] > 0),
    "POS_RISK_GROUP"
] = "History with overdue"

# Compare repayment difficulty rates
pos_risk = (
    pos_analysis.groupby("POS_RISK_GROUP")["TARGET"]
    .agg(["count", "mean"])
)

pos_risk["mean"] *= 100

print("Repayment difficulty by POS/Cash history:")
print(pos_risk)

print("\nTotal applicants:", len(pos_analysis))


Repayment difficulty by POS/Cash history:
                      count       mean
POS_RISK_GROUP                        
History with overdue   4845  10.051600
History, no overdue   20549   7.718137
No POS/Cash history    1606   6.662516

Total applicants: 27000


### Finding — POS/Cash Payment History

The POS/Cash dataset contains 754,688 monthly records covering 25,394 applicants. Historical overdue payments were identified in 2.92% of monthly records.

Applicants with at least one recorded overdue payment had a repayment difficulty rate of **10.05%**, compared with **7.72%** for applicants with POS/Cash history but no overdue payments. Applicants without POS/Cash history had a rate of **6.66%**.

These findings suggest that historical payment delays may be useful predictors of repayment difficulty. Applicant-level features such as overdue frequency, overdue rate and maximum days past due should be considered during feature engineering and evaluated using validation data.

## 7. Credit Card Balance Analysis

This section explores historical credit card balances, including credit utilisation, repayment behaviour and overdue payments.

The objective is to assess the structure and coverage of the dataset and identify potential credit risk features for subsequent modelling.

In [32]:

# Load credit card balance dataset
credit_card = pd.read_csv("/content/credit_card_balance.csv")

print("Dataset shape:", credit_card.shape)

print("\nUnique applicants:",
      credit_card["SK_ID_CURR"].nunique())

print("\nColumn names:")
print(credit_card.columns.tolist())

print("\nFirst five rows:")
display(credit_card.head())


Dataset shape: (281545, 23)

Unique applicants: 7596

Column names:
['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']

First five rows:


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0
1,1235299,203885,-5,201261.195,225000,76500.0,111026.7,0.0,34526.7,6338.340,...,197224.695,197224.695,3.0,9,0.0,6.0,38.0,Active,0,0
2,1441883,171537,-5,0.000,270000,0.0,0.0,0.0,0.0,12218.490,...,0.000,0.000,0.0,0,0.0,0.0,2.0,Active,0,0
3,1009803,313797,-4,135326.115,225000,0.0,0.0,0.0,0.0,13500.000,...,135326.115,135326.115,0.0,0,0.0,0.0,95.0,Active,0,0
4,2293769,352647,-4,110310.120,450000,0.0,0.0,0.0,0.0,6054.075,...,110310.120,110310.120,0.0,0,0.0,0.0,85.0,Active,0,0


In [33]:
# Check missing values in key credit card variables
columns = [
    "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL",
    "SK_DPD",
    "SK_DPD_DEF"
]

print("Missing values (%):")
print((credit_card[columns].isna().mean() * 100).round(2))

# Credit card balance statistics
print("\nCredit card balance statistics:")
print(credit_card["AMT_BALANCE"].describe())

# Credit limit statistics
print("\nCredit limit statistics:")
print(credit_card["AMT_CREDIT_LIMIT_ACTUAL"].describe())

# Overdue credit card records
overdue_records = (credit_card["SK_DPD"] > 0).sum()

print("\nRecords with overdue payments:", overdue_records)
print(
    "Percentage of records with overdue payments:",
    round(overdue_records / len(credit_card) * 100, 2),
    "%"
)

Missing values (%):
AMT_BALANCE                0.0
AMT_CREDIT_LIMIT_ACTUAL    0.0
SK_DPD                     0.0
SK_DPD_DEF                 0.0
dtype: float64

Credit card balance statistics:
count    2.815450e+05
mean     5.906682e+04
std      1.079116e+05
min     -8.592417e+04
25%      0.000000e+00
50%      0.000000e+00
75%      8.960085e+04
max      1.139278e+06
Name: AMT_BALANCE, dtype: float64

Credit limit statistics:
count    2.815450e+05
mean     1.532841e+05
std      1.658644e+05
min      0.000000e+00
25%      4.500000e+04
50%      1.125000e+05
75%      1.800000e+05
max      1.237500e+06
Name: AMT_CREDIT_LIMIT_ACTUAL, dtype: float64

Records with overdue payments: 12268
Percentage of records with overdue payments: 4.36 %


In [34]:
# Identify overdue credit card records
credit_card["HAS_OVERDUE"] = credit_card["SK_DPD"] > 0

# Aggregate monthly records to applicant level
card_features = (
    credit_card.groupby("SK_ID_CURR")
    .agg(
        CARD_TOTAL_RECORDS=("SK_DPD", "size"),
        CARD_OVERDUE_RECORDS=("HAS_OVERDUE", "sum"),
        CARD_MAX_DPD=("SK_DPD", "max"),
        CARD_AVG_BALANCE=("AMT_BALANCE", "mean"),
        CARD_AVG_LIMIT=("AMT_CREDIT_LIMIT_ACTUAL", "mean")
    )
    .reset_index()
)

# Calculate percentage of overdue records
card_features["CARD_OVERDUE_RATE"] = (
    card_features["CARD_OVERDUE_RECORDS"]
    / card_features["CARD_TOTAL_RECORDS"] * 100
)

print("Applicants with credit card history:", len(card_features))

print("\nFirst five applicants:")
display(card_features.head())

Applicants with credit card history: 7596

First five applicants:


,SK_ID_CURR,CARD_TOTAL_RECORDS,CARD_OVERDUE_RECORDS,CARD_MAX_DPD,CARD_AVG_BALANCE,CARD_AVG_LIMIT,CARD_OVERDUE_RATE
0,100048,31,0,0,59810.556774,225000.000000,0.000000
1,100100,94,12,1,40102.360851,81861.702128,12.765957
2,100144,42,0,0,0.000000,180000.000000,0.000000
3,100153,73,8,1,121103.898288,157500.000000,10.958904
4,100157,45,0,0,0.000000,291500.000000,0.000000


In [35]:
# Merge credit card features with main applications
card_analysis = applications[
    ["SK_ID_CURR", "TARGET"]
].copy()

card_analysis = card_analysis.merge(
    card_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

# Classify applicants by credit card history
card_analysis["CARD_RISK_GROUP"] = "No credit card history"

has_history = card_analysis["CARD_TOTAL_RECORDS"].notna()

card_analysis.loc[
    has_history & (card_analysis["CARD_OVERDUE_RECORDS"] == 0),
    "CARD_RISK_GROUP"
] = "History, no overdue"

card_analysis.loc[
    has_history & (card_analysis["CARD_OVERDUE_RECORDS"] > 0),
    "CARD_RISK_GROUP"
] = "History with overdue"

# Compare repayment difficulty rates
card_risk = (
    card_analysis.groupby("CARD_RISK_GROUP")["TARGET"]
    .agg(["count", "mean"])
)

card_risk["mean"] *= 100

print("Repayment difficulty by credit card history:")
print(card_risk)

print("\nTotal applicants:", len(card_analysis))

Repayment difficulty by credit card history:
                        count      mean
CARD_RISK_GROUP                        
History with overdue     1599  7.191995
History, no overdue      5997  8.987827
No credit card history  19404  7.864358

Total applicants: 27000


### Finding 7 — Credit Card Repayment History

Credit card history is available for 7,596 out of 27,000 applicants (28.1%). Applicants with recorded credit card overdue payments have a repayment difficulty rate of 7.19%, compared with 8.99% for those with credit card history but no recorded overdue payments.

Unlike other repayment history datasets, credit card overdue records do not show a positive association with repayment difficulty in this sample. This suggests that credit card features should be evaluated through predictive modelling rather than assumed to be strong risk indicators.

## 8. Data Understanding Summary

### 8.1 Dataset Overview

The analysis examined seven relational datasets containing applicant characteristics, previous loan applications, external credit records, monthly credit histories, installment payments, POS/Cash balances, and credit card balances.

The main development dataset contains **27,000 applicants**, with `TARGET` indicating whether an applicant experienced repayment difficulties.

Historical datasets contain multiple records per applicant. Therefore, customer-level aggregation using `SK_ID_CURR` is required before combining these datasets for predictive modelling.

### 8.2 Key Findings

**Previous Loan Applications**

Applicants with previous loan refusals showed higher repayment difficulty rates. The rate increased from approximately 7.0% among applicants with no previous refusals to 14.5% among those with six or more refusals.

**External Credit Accounts**

Applicants with recorded overdue external credit accounts had a repayment difficulty rate of 16.7%, compared with 7.6% among applicants with external credit records but no recorded overdue accounts.

**External Account History**

Historical delinquency was associated with higher repayment difficulty: 10.7% among applicants with recorded delinquency versus 7.5% among those with monthly history but no recorded delinquency.

**Installment Payments**

Repayment difficulty generally increased with the proportion of late installment payments, from 6.8% among applicants with no late payments to 13.5% among those with late payments in more than 30% of recorded installments.

**POS/Cash Balance**

Applicants with recorded POS/Cash overdue payments had a repayment difficulty rate of 10.1%, compared with 7.7% among those with POS/Cash history but no recorded overdue payments.

**Credit Card Balance**

Credit card overdue records showed a different pattern. Applicants with recorded overdue payments had a repayment difficulty rate of 7.2%, compared with 9.0% among those with credit card history but no recorded overdue payments.

### 8.3 Implications for Predictive Modelling

The exploratory analysis suggests that historical repayment behaviour, previous loan refusals, and external credit information may provide useful predictive features.

Potential engineered features include:

- Previous loan refusal count and refusal rate.
- Number of external credit accounts and recorded overdue accounts.
- Historical delinquency counts.
- Late installment payment rate and maximum payment delay.
- POS/Cash overdue rate and maximum days past due.
- Credit card overdue rate, average balance, and average credit limit.

These relationships are descriptive rather than causal. Their predictive usefulness must be assessed using out-of-sample validation.

### 8.4 Next Steps

The next stage follows the CRISP-DM Data Preparation phase:

1. Aggregate historical records to applicant level.
2. Merge engineered features into the main application dataset.
3. Investigate missing values and data quality.
4. Prepare numerical and categorical features.
5. Create stratified training and validation datasets.
6. Develop and evaluate classification models.